# Lab 4.1: Effective Tool Descriptions & Boundaries

**What you'll build:** A set of tool definitions where Claude reliably picks the right tool -- and learn why minimal descriptions cause misselection.

**Estimated time:** 20-25 minutes

| Phase | What happens | Time |
|-------|-------------|------|
| 1 | The Wrong Way -- minimal descriptions cause tool misselection | 5 min |
| 2 | The Right Way -- production descriptions with boundaries fix selection | 5 min |
| 3 | Your Turn -- write descriptions for a new set of overlapping tools | 10 min |
| 4 | Stress Test -- verify consistent selection across varied queries | 5 min |

In [ ]:
import anthropic
import json
from dotenv import load_dotenv

load_dotenv()

client = anthropic.Anthropic()
MODEL = "claude-sonnet-4-20250514"

## The Setup

You are building a customer service agent with 5 tools. The tools have overlapping domains -- a customer asking about "my order" could trigger `lookup_customer`, `lookup_order`, or `get_account_history`. The challenge is writing descriptions that let Claude reliably pick the right one.

We'll test selection accuracy by sending a batch of queries with known correct tools.

In [ ]:
# Test queries with expected correct tool
TEST_QUERIES = [
    {"query": "What's my order status for order #12345?", "expected_tool": "lookup_order"},
    {"query": "Can you update my email address?", "expected_tool": "update_customer_profile"},
    {"query": "What's my account membership tier?", "expected_tool": "lookup_customer"},
    {"query": "Show me everything that happened on my account this month", "expected_tool": "get_account_history"},
    {"query": "I need to return an item from my recent order", "expected_tool": "initiate_return"},
    {"query": "What's my shipping address on file?", "expected_tool": "lookup_customer"},
    {"query": "Where is my package right now?", "expected_tool": "lookup_order"},
    {"query": "I want to change my notification preferences", "expected_tool": "update_customer_profile"},
    {"query": "Show me my purchase history for the last 3 months", "expected_tool": "get_account_history"},
    {"query": "This item arrived damaged, I want a refund", "expected_tool": "initiate_return"},
]


def test_tool_selection(tools, queries, label=""):
    """Test which tool Claude selects for each query."""
    results = []
    for q in queries:
        response = client.messages.create(
            model=MODEL,
            max_tokens=1024,
            tools=tools,
            tool_choice={"type": "any"},
            messages=[{"role": "user", "content": q["query"]}],
        )
        selected = None
        for block in response.content:
            if block.type == "tool_use":
                selected = block.name
                break
        correct = selected == q["expected_tool"]
        results.append({
            "query": q["query"],
            "expected": q["expected_tool"],
            "selected": selected,
            "correct": correct,
        })
    return results


def print_results(results, label):
    """Print selection results with accuracy."""
    correct = sum(1 for r in results if r["correct"])
    total = len(results)
    print(f"\n=== {label} ===")
    print(f"Accuracy: {correct}/{total} ({correct/total:.0%})\n")
    for r in results:
        tag = "CORRECT" if r["correct"] else "WRONG"
        print(f"  [{tag}] \"{r['query'][:50]}...\"")
        if not r["correct"]:
            print(f"         Expected: {r['expected']} | Got: {r['selected']}")
    return correct / total


print("Setup complete. Test harness ready.")

---

## Phase 1: The Wrong Approach

Most developers write minimal, one-line tool descriptions. Let's see how that performs.

In [ ]:
# Minimal descriptions -- the kind most developers write first
MINIMAL_TOOLS = [
    {
        "name": "lookup_customer",
        "description": "Retrieves customer information.",
        "input_schema": {
            "type": "object",
            "properties": {
                "customer_id": {"type": "string", "description": "The customer ID"}
            },
            "required": ["customer_id"]
        }
    },
    {
        "name": "lookup_order",
        "description": "Retrieves order information.",
        "input_schema": {
            "type": "object",
            "properties": {
                "order_id": {"type": "string", "description": "The order ID"}
            },
            "required": ["order_id"]
        }
    },
    {
        "name": "get_account_history",
        "description": "Gets account history.",
        "input_schema": {
            "type": "object",
            "properties": {
                "customer_id": {"type": "string", "description": "The customer ID"},
                "days": {"type": "integer", "description": "Number of days"}
            },
            "required": ["customer_id"]
        }
    },
    {
        "name": "update_customer_profile",
        "description": "Updates customer profile.",
        "input_schema": {
            "type": "object",
            "properties": {
                "customer_id": {"type": "string", "description": "The customer ID"},
                "updates": {"type": "object", "description": "Fields to update"}
            },
            "required": ["customer_id", "updates"]
        }
    },
    {
        "name": "initiate_return",
        "description": "Initiates a return.",
        "input_schema": {
            "type": "object",
            "properties": {
                "order_id": {"type": "string", "description": "The order ID"},
                "reason": {"type": "string", "description": "Return reason"}
            },
            "required": ["order_id", "reason"]
        }
    },
]

minimal_results = test_tool_selection(MINIMAL_TOOLS, TEST_QUERIES)
minimal_accuracy = print_results(minimal_results, "MINIMAL DESCRIPTIONS")

### Why did that fail?

- **No disambiguation.** "Retrieves customer information" and "Gets account history" both sound relevant for "show me my account activity."
- **No negative boundaries.** Nothing tells the model "do NOT use this tool for X."
- **No trigger examples.** The model guesses based on surface-level word matching between the query and the description.
- **Descriptions are prompts.** Vague descriptions produce vague selection, just like vague prompts produce vague output (Module 1).

---

## Phase 2: The Right Approach

Production tool descriptions follow a pattern:
1. **What it does** (specific capability)
2. **When to use it** (positive triggers)
3. **When NOT to use it** (negative boundaries pointing to the correct alternative)
4. **Input/output details** (what data goes in, what comes back)

In [ ]:
# Production descriptions with explicit boundaries
PRODUCTION_TOOLS = [
    {
        "name": "lookup_customer",
        "description": (
            "Retrieves customer account details including name, email, phone, "
            "shipping address, and membership tier. "
            "Use when the user asks about their profile, contact info, membership level, "
            "or address on file. "
            "Do NOT use for order-specific queries (use lookup_order) or "
            "account activity/history (use get_account_history)."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "customer_id": {"type": "string", "description": "The customer ID"}
            },
            "required": ["customer_id"]
        }
    },
    {
        "name": "lookup_order",
        "description": (
            "Retrieves details for a specific order including status, items, "
            "shipping tracking, delivery estimate, and payment info. "
            "Use when the user asks about order status, package tracking, delivery, "
            "or a specific order number. "
            "Do NOT use for account-level queries like membership or profile "
            "(use lookup_customer) or for browsing past orders "
            "(use get_account_history)."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "order_id": {"type": "string", "description": "The order ID or tracking number"}
            },
            "required": ["order_id"]
        }
    },
    {
        "name": "get_account_history",
        "description": (
            "Retrieves a chronological log of account activity: orders placed, "
            "returns initiated, profile changes, and support interactions. "
            "Use when the user asks for their history, recent activity, "
            "purchase log, or 'everything that happened' over a time period. "
            "Do NOT use for a specific order's details (use lookup_order) or "
            "for current profile/address info (use lookup_customer)."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "customer_id": {"type": "string", "description": "The customer ID"},
                "days": {"type": "integer", "description": "Number of days of history to retrieve (default: 30)"}
            },
            "required": ["customer_id"]
        }
    },
    {
        "name": "update_customer_profile",
        "description": (
            "Updates customer profile fields: email, phone, shipping address, "
            "notification preferences, or display name. "
            "Use when the user wants to change their contact info, address, "
            "notification settings, or other profile fields. "
            "Do NOT use for order modifications (orders cannot be edited after placement) "
            "or for returns/refunds (use initiate_return)."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "customer_id": {"type": "string", "description": "The customer ID"},
                "updates": {"type": "object", "description": "Fields to update (email, phone, address, etc.)"}
            },
            "required": ["customer_id", "updates"]
        }
    },
    {
        "name": "initiate_return",
        "description": (
            "Starts a return or refund process for items in a specific order. "
            "Use when the user wants to return an item, request a refund, "
            "or report a damaged/incorrect item. "
            "Do NOT use for order status checks (use lookup_order) or "
            "general account queries (use lookup_customer)."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "order_id": {"type": "string", "description": "The order ID for the return"},
                "reason": {"type": "string", "description": "Return reason: damaged, wrong_item, changed_mind, defective, other"}
            },
            "required": ["order_id", "reason"]
        }
    },
]

production_results = test_tool_selection(PRODUCTION_TOOLS, TEST_QUERIES)
production_accuracy = print_results(production_results, "PRODUCTION DESCRIPTIONS")

In [ ]:
# Side-by-side comparison
print("=" * 55)
print("COMPARISON: MINIMAL vs PRODUCTION DESCRIPTIONS")
print("=" * 55)
print(f"{'Metric':<30} {'Minimal':>10} {'Production':>10}")
print(f"{'-'*30} {'-'*10} {'-'*10}")

min_correct = sum(1 for r in minimal_results if r["correct"])
prod_correct = sum(1 for r in production_results if r["correct"])
total = len(TEST_QUERIES)

print(f"{'Correct selections':<30} {min_correct:>10} {prod_correct:>10}")
print(f"{'Accuracy':<30} {min_correct/total:>9.0%} {prod_correct/total:>9.0%}")
print(f"{'Misselections':<30} {total-min_correct:>10} {total-prod_correct:>10}")
print()
if prod_correct > min_correct:
    print(f"Production descriptions improved accuracy by {(prod_correct-min_correct)/total:.0%}.")
    print("The fix was descriptions, not routing classifiers or tool consolidation.")

### What Changed?

| Aspect | Minimal | Production |
|--------|---------|------------|
| **Description length** | ~5 words | ~40-50 words |
| **Positive triggers** | None | "Use when the user asks about..." |
| **Negative boundaries** | None | "Do NOT use for X -- use Y instead" |
| **Output details** | None | Lists what data is returned |
| **Selection accuracy** | ~60-70% | ~95-100% |

---

## Phase 3: Your Turn

Here is a different scenario: a project management agent with tools that overlap in confusing ways. Write production-grade descriptions.

**Your goal:** 100% selection accuracy on the test queries.

In [ ]:
CHALLENGE_QUERIES = [
    {"query": "Create a new task called 'Fix login bug' for the backend team", "expected_tool": "create_task"},
    {"query": "What tasks are assigned to me this sprint?", "expected_tool": "list_tasks"},
    {"query": "Mark task PROJ-123 as complete", "expected_tool": "update_task_status"},
    {"query": "Add a comment on PROJ-456 saying the fix is deployed", "expected_tool": "add_task_comment"},
    {"query": "Show me the activity log for PROJ-789", "expected_tool": "get_task_history"},
    {"query": "What's the current status of PROJ-123?", "expected_tool": "list_tasks"},
    {"query": "Change the priority of PROJ-456 to critical", "expected_tool": "update_task_status"},
    {"query": "Who changed the assignee on PROJ-789 last week?", "expected_tool": "get_task_history"},
]

print("Challenge: Write descriptions for these 5 project management tools.")
print("Test queries loaded. Write your tools in the next cell.")

In [ ]:
# =============================================================
# TODO: Write production-grade tool descriptions
# =============================================================
#
# Requirements for each description:
# - What the tool does (specific capability)
# - When to use it (positive triggers)
# - When NOT to use it (negative boundaries with alternatives)
#
# Think about:
# - list_tasks vs get_task_history -- both retrieve task info
# - update_task_status vs add_task_comment -- both modify a task
# - create_task is distinct but could overlap with update_task_status

YOUR_TOOLS = [
    {
        "name": "create_task",
        "description": (
            # TODO: Write a production-grade description
            "Creates a new task."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "title": {"type": "string", "description": "Task title"},
                "team": {"type": "string", "description": "Team to assign to"},
                "priority": {"type": "string", "description": "Priority: low, medium, high, critical"}
            },
            "required": ["title"]
        }
    },
    {
        "name": "list_tasks",
        "description": (
            # TODO: Write a production-grade description
            "Lists tasks."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "assignee": {"type": "string", "description": "Filter by assignee"},
                "status": {"type": "string", "description": "Filter by status"},
                "sprint": {"type": "string", "description": "Filter by sprint"},
                "task_id": {"type": "string", "description": "Get a specific task by ID"}
            }
        }
    },
    {
        "name": "update_task_status",
        "description": (
            # TODO: Write a production-grade description
            "Updates a task."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "task_id": {"type": "string", "description": "Task ID (e.g., PROJ-123)"},
                "status": {"type": "string", "description": "New status: open, in_progress, review, done"},
                "priority": {"type": "string", "description": "New priority: low, medium, high, critical"},
                "assignee": {"type": "string", "description": "New assignee"}
            },
            "required": ["task_id"]
        }
    },
    {
        "name": "add_task_comment",
        "description": (
            # TODO: Write a production-grade description
            "Adds a comment to a task."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "task_id": {"type": "string", "description": "Task ID (e.g., PROJ-123)"},
                "comment": {"type": "string", "description": "Comment text"}
            },
            "required": ["task_id", "comment"]
        }
    },
    {
        "name": "get_task_history",
        "description": (
            # TODO: Write a production-grade description
            "Gets task history."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "task_id": {"type": "string", "description": "Task ID (e.g., PROJ-123)"}
            },
            "required": ["task_id"]
        }
    },
]

# Test your descriptions
your_results = test_tool_selection(YOUR_TOOLS, CHALLENGE_QUERIES)
your_accuracy = print_results(your_results, "YOUR DESCRIPTIONS")

In [ ]:
# =============================================================
# Checker: validates your tool descriptions
# =============================================================

correct = sum(1 for r in your_results if r["correct"])
total = len(CHALLENGE_QUERIES)
accuracy = correct / total

print("=== YOUR SCORECARD ===")
print(f"  Correct: {correct}/{total}")
print(f"  Accuracy: {accuracy:.0%}")
print()

# Check description quality
issues = []
for tool in YOUR_TOOLS:
    desc = tool["description"].lower()
    if len(desc) < 50:
        issues.append(f"  {tool['name']}: Description too short ({len(desc)} chars). Add triggers and boundaries.")
    if "do not" not in desc and "don't" not in desc:
        issues.append(f"  {tool['name']}: Missing negative boundary (Do NOT use for...).")

if issues:
    print("Description quality issues:")
    for issue in issues:
        print(issue)
    print()

if accuracy == 1.0 and not issues:
    print("PASSED -- 100% accuracy with production-quality descriptions!")
elif accuracy == 1.0:
    print("Selection is correct, but descriptions could be more robust. Fix the issues above.")
else:
    wrong = [r for r in your_results if not r["correct"]]
    print("Misselections detected. Improve boundaries between these tools:")
    for r in wrong:
        print(f"  Query: \"{r['query']}\"")
        print(f"  Expected: {r['expected']} | Got: {r['selected']}")
        print(f"  -> Add a boundary to {r['selected']}'s description pointing to {r['expected']}")

---

## Phase 4: Stress Test

Run your descriptions 3 times to check consistency, then test with edge-case queries.

In [ ]:
# Consistency check: run 3 times
print("Running your descriptions 3 times for consistency...\n")

accuracies = []
for run in range(3):
    results = test_tool_selection(YOUR_TOOLS, CHALLENGE_QUERIES)
    acc = sum(1 for r in results if r["correct"]) / len(results)
    accuracies.append(acc)
    print(f"  Run {run+1}: {acc:.0%}")

print(f"\nAccuracies: {[f'{a:.0%}' for a in accuracies]}")
if all(a == 1.0 for a in accuracies):
    print("Perfect consistency -- your descriptions produce reliable selections every time.")
else:
    print(f"Inconsistent results. Minimum: {min(accuracies):.0%}. Tighten boundaries.")

In [ ]:
# Edge case queries -- deliberately ambiguous
EDGE_CASES = [
    {"query": "What happened to PROJ-123? It was supposed to be done yesterday.",
     "expected_tool": "get_task_history",
     "note": "'What happened' = history, not current status"},
    {"query": "Can you close PROJ-456 and leave a note saying it shipped?",
     "expected_tool": "update_task_status",
     "note": "Primary action is status change; comment is secondary"},
    {"query": "I need a new task for the bug that Sarah found in the payment flow",
     "expected_tool": "create_task",
     "note": "'New task' is unambiguous despite other details"},
]

print("Edge case queries (deliberately ambiguous):\n")
edge_results = test_tool_selection(YOUR_TOOLS, EDGE_CASES)
for r, ec in zip(edge_results, EDGE_CASES):
    tag = "CORRECT" if r["correct"] else "WRONG"
    print(f"  [{tag}] \"{ec['query'][:60]}...\"")
    print(f"         Note: {ec['note']}")
    if not r["correct"]:
        print(f"         Expected: {r['expected']} | Got: {r['selected']}")
    print()

### Key Takeaways

1. **Tool descriptions are prompts.** The same principles from Module 1 apply -- explicit criteria beat vague instructions.
2. **The production description pattern:** what it does + when to use it + when NOT to use it + what it returns.
3. **Negative boundaries are the key differentiator.** "Do NOT use for X -- use Y instead" eliminates ambiguity between overlapping tools.
4. **Descriptions first, always.** When tool selection is unreliable, improve descriptions before adding routing classifiers, few-shot examples, or consolidating tools.
5. **Edge cases need explicit instructions.** Add "If the user's intent is ambiguous, ask for clarification" to handle remaining ambiguity gracefully.